# Project Integration: Fine-Tuned Resume Agent — Week 5

**Notebook:** `08_project_integration.ipynb`  
**Estimated time:** 30 minutes  

## Objectives
1. Wrap the fine-tuned model as a **gbrain-style skill** using `src.skill_wrapper`
2. Integrate HW4's RAG pipeline for retrieval-augmented generation
3. Run 5 end-to-end queries through the full agent pipeline
4. Author a `RESOLVER.md` documenting when to route to this skill

## Prerequisites
- NB07 complete: `outputs/merged_model/` or Ollama `hw5-finetuned` model registered
- (Optional) HW4 RAG pipeline at `../../Homework4-Submission/src/rag_pipeline.py`

---
## Introduction

In **Week 4** we built a RAG pipeline that retrieves relevant chunks from a PDF resume and passes them to an LLM. In **Week 5** we fine-tuned a model (`Qwen2.5-0.5B-Instruct`) to respond in a consistent, confident tone about resume content.

Now we combine them into a **gbrain-inspired agent skill**:

```
User query
    │
    ▼
RESOLVER (SkillResolver)
    │  routes based on keywords / intent
    ▼
hw5-resume-skill
    │  + retrieval context from RAG (if available)
    ▼
Fine-tuned Qwen2.5-0.5B (hw5-finetuned via Ollama)
    │
    ▼
Answer
```

This is the **gbrain architecture pattern**: `RESOLVER → skill dispatch → retrieval augmentation`. Each skill is a specialized model or tool; the resolver picks the right one based on the query. In production gbrain, skills are registered via `RESOLVER.md` manifests — we'll create one at the end of this notebook.

In [1]:
import sys
import importlib
import os
import json

sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
tracker = CostTracker()

print("Setup complete.")

Setup complete.


---
## Part 1: Building the Skill

A **skill** in gbrain is any callable that takes a `(prompt, context)` pair and returns a string answer. We wrap our fine-tuned model in a `FineTunedSkill` object that also carries metadata for the resolver: a name, a description, and keyword triggers.

In [2]:
import importlib
import src.skill_wrapper as _sw
importlib.reload(_sw)

from src.skill_wrapper import FineTunedSkill, SkillResolver, make_resume_skill
from src.llm_client import LLMClient

# Try to use the fine-tuned Ollama model; fall back to qwen3.5:27b
try:
    import ollama
    # Quick test to see if hw5-finetuned is registered
    _test = ollama.chat(
        model="hw5-finetuned",
        messages=[{"role": "user", "content": "ping"}],
    )
    model_fn = lambda prompt: ollama.chat(
        model="hw5-finetuned",
        messages=[{"role": "user", "content": prompt}],
    )["message"]["content"]
    print("Using fine-tuned hw5-finetuned model via Ollama")
except Exception as e:
    print(f"hw5-finetuned not available ({e}). Falling back to qwen3.5:27b via Ollama.")
    llm_client = LLMClient(path="B")  # Ollama fallback
    model_fn = lambda prompt: llm_client.generate(prompt)["content"]
    print("Fallback: using qwen3.5:27b")

# Build the resume skill
resume_skill = make_resume_skill(model_fn)

# Register skills with the resolver
resolver = SkillResolver([resume_skill])
resolver.show_skills()

Using fine-tuned hw5-finetuned model via Ollama
[skill_wrapper] Registered skill 'resume_qa' with 19 keywords
[skill_wrapper] SkillResolver initialized with 1 skill(s)
[skill_wrapper] Registered skills (1 total):
  [1] resume_qa
       Description: Answers questions about professional experience, skills, education, and career history from a resume.
       Keywords: resume, experience, skills, education, career, job, work, background, qualification, degree, university, company, position, role, project, achievement, certification, profile, candidate


---
## Part 2: Adding RAG Context (gbrain Pattern)

In gbrain, every skill can receive **retrieval context** from the knowledge graph. We replicate this pattern using our HW4 RAG modules: when a user asks a resume question, we first retrieve the top-3 relevant chunks from the PDF, then pass them as context to the fine-tuned model.

This is more powerful than either approach alone:
- **RAG alone** gives factual grounding but generic phrasing
- **Fine-tuning alone** gives the right tone but may hallucinate facts
- **RAG + fine-tuning** gives accurate facts in the right tone

In [3]:
# Try to load HW4 RAG pipeline
sys.path.insert(0, '../../Homework4-Submission')

try:
    from src.rag_pipeline import RAGPipeline
    rag = RAGPipeline.from_pdf("../test_data/sample_resume.pdf")
    has_rag = True
    print("HW4 RAG pipeline loaded successfully")
except Exception as e:
    has_rag = False
    print(f"RAG not available: {e}")
    print("Continuing in skill-only mode (no retrieval context).")


def query_with_rag(question: str) -> str:
    """Route a question through the resolver, optionally enriching with RAG context."""
    if has_rag:
        ctx = rag.retrieve(question, top_k=3)
        retrieval_ctx = "\n".join([c["text"] for c in ctx])
    else:
        retrieval_ctx = ""
    return resolver.dispatch(question, retrieval_ctx=retrieval_ctx)


print(f"\nquery_with_rag ready (RAG enabled: {has_rag})")

RAG not available: No module named 'src.rag_pipeline'
Continuing in skill-only mode (no retrieval context).

query_with_rag ready (RAG enabled: False)


---
## Part 3: End-to-End Agent Run

Let's run 5 representative resume queries through the full pipeline and inspect the answers.

In [4]:
test_queries = [
    "What programming languages does Scott know?",
    "What was Scott's most recent job?",
    "Does Scott have experience with machine learning?",
    "What is Scott's educational background?",
    "What kind of roles is Scott looking for?",
]

print(f"Running {len(test_queries)} queries through the full agent pipeline...")
print("=" * 70)

for i, q in enumerate(test_queries, 1):
    print(f"\n[Query {i}/{len(test_queries)}]")
    print(f"Q: {q}")
    answer = query_with_rag(q)
    preview = answer[:200]
    print(f"A: {preview}{'...' if len(answer) > 200 else ''}")
    print("-" * 70)

Running 5 queries through the full agent pipeline...

[Query 1/5]
Q: What programming languages does Scott know?
[skill_wrapper] Resolving query: What programming languages does Scott know?...
[skill_wrapper] No keyword match, falling back to 'resume_qa'
[skill_wrapper] Skill 'resume_qa' invoked for query: What programming languages does Scott know?...
[skill_wrapper] Skill 'resume_qa' returned 196 chars
A: Scott is proficient in Python, TypeScript, and SQL. He uses Python for all ML/data work and TypeScript for full-stack applications. He also has experience with bash scripting for DevOps workflows.
----------------------------------------------------------------------

[Query 2/5]
Q: What was Scott's most recent job?
[skill_wrapper] Resolving query: What was Scott's most recent job?...
[skill_wrapper] Resolved to skill 'resume_qa' (score=1.0)
[skill_wrapper] Skill 'resume_qa' invoked for query: What was Scott's most recent job?...
[skill_wrapper] Skill 'resume_qa' returned 158 chars


---
## Part 4: RESOLVER.md — gbrain Documentation Pattern

In gbrain, each skill is documented in a `RESOLVER.md` manifest. This file tells the orchestrator:
- **When** to route to this skill (what kinds of questions it handles)
- **What keywords** signal this skill is relevant
- **Example queries** for few-shot routing
- **Fallback** behavior when no skill matches

Let's generate one for our resume skill:

In [5]:
resolver_md = """# RESOLVER.md — Skill Routing Guide

## hw5-resume-skill
**When to use:** Queries about resume content, work history, skills, education, career goals  
**Keywords:** resume, experience, job, background, skills, education, worked, built, achieved  
**Example queries:**
- "What languages does Scott know?"
- "Tell me about Scott's experience at [company]"
- "What is Scott's highest degree?"
- "What projects has Scott built?"
- "What kind of roles is Scott targeting?"

**Model:** hw5-finetuned (Qwen2.5-0.5B-Instruct, fine-tuned on synthetic resume Q&A)  
**RAG:** Yes — retrieves top-3 chunks from sample_resume.pdf before generating  

## Fallback
If no skill matches, route to the base LLM (qwen3.5:27b via Ollama).  
Trigger condition: query does not contain resume-related keywords AND no skill confidence > 0.5.
"""

os.makedirs("outputs", exist_ok=True)
with open("outputs/RESOLVER.md", "w") as f:
    f.write(resolver_md)

print(resolver_md)

# RESOLVER.md — Skill Routing Guide

## hw5-resume-skill
**When to use:** Queries about resume content, work history, skills, education, career goals  
**Keywords:** resume, experience, job, background, skills, education, worked, built, achieved  
**Example queries:**
- "What languages does Scott know?"
- "Tell me about Scott's experience at [company]"
- "What is Scott's highest degree?"
- "What projects has Scott built?"
- "What kind of roles is Scott targeting?"

**Model:** hw5-finetuned (Qwen2.5-0.5B-Instruct, fine-tuned on synthetic resume Q&A)  
**RAG:** Yes — retrieves top-3 chunks from sample_resume.pdf before generating  

## Fallback
If no skill matches, route to the base LLM (qwen3.5:27b via Ollama).  
Trigger condition: query does not contain resume-related keywords AND no skill confidence > 0.5.



---
## TODO 1: Add a Second Skill

The `SkillResolver` currently has only one skill (`hw5-resume-skill`). Add a **"general Q&A" skill** that:
- Uses the Ollama base model (`qwen3.5:27b`) for any query the resume skill doesn't match
- Has different keyword triggers (e.g., "explain", "what is", "how does")
- Is registered with the resolver alongside the resume skill

Then test that the resolver routes correctly:
- `"What is Scott's background?"` → resume skill
- `"What is gradient descent?"` → general Q&A skill

**Starter code:**

In [6]:
# TODO 1: Add a general Q&A skill and test routing

# Example:
# general_client = LLMClient(path="B")  # Ollama qwen3.5:27b
# general_fn = lambda prompt: general_client.generate(prompt)["content"]
#
# general_skill = FineTunedSkill(
#     name="general-qa",
#     description="Answers general knowledge questions not related to the resume",
#     model_fn=general_fn,
#     keywords=["explain", "what is", "how does", "why", "define", "describe"],
# )
#
# multi_resolver = SkillResolver([resume_skill, general_skill])
# multi_resolver.show_skills()
#
# # Test routing
# print(multi_resolver.dispatch("What is Scott's educational background?"))
# print(multi_resolver.dispatch("What is gradient descent?"))

todo1_reflection = "[TODO 1: Fill in your second-skill routing test results here]"
print(todo1_reflection)

[TODO 1: Fill in your second-skill routing test results here]


---
## TODO 2: Project Update

Write your weekly project update in the cell below, then run the summary cell to save it.

**Template:**
```markdown
# Week 5 Project Update — [Your Name]
## What I built this week
## How Week 5 connects to Week 4 (RAG + fine-tuning)
## What surprised me most about fine-tuning
## What I would improve with more compute/time
```

In [7]:
# TODO 2: Fill in your project update
project_update = """
# Week 5 Project Update — [Your Name]

## What I built this week
[TODO: Describe what you built — e.g., fine-tuned Qwen2.5-0.5B, deployed via Ollama, built the skill resolver]

## How Week 5 connects to Week 4 (RAG + fine-tuning)
[TODO: Explain how HW4 RAG retrieves context and HW5 fine-tuned model generates tailored answers]

## What surprised me most about fine-tuning
[TODO: Fill in your genuine reaction — something unexpected about the training process, eval results, or deployment]

## What I would improve with more compute/time
[TODO: Longer training, more data, better base model, RLHF, multi-skill resolver, etc.]
"""

print(project_update)


# Week 5 Project Update — [Your Name]

## What I built this week
[TODO: Describe what you built — e.g., fine-tuned Qwen2.5-0.5B, deployed via Ollama, built the skill resolver]

## How Week 5 connects to Week 4 (RAG + fine-tuning)
[TODO: Explain how HW4 RAG retrieves context and HW5 fine-tuned model generates tailored answers]

## What surprised me most about fine-tuning
[TODO: Fill in your genuine reaction — something unexpected about the training process, eval results, or deployment]

## What I would improve with more compute/time
[TODO: Longer training, more data, better base model, RLHF, multi-skill resolver, etc.]



---
## Summary

In [8]:
from datetime import datetime

# Save project update
os.makedirs("outputs", exist_ok=True)
update_content = project_update.strip() if 'project_update' in dir() else "[TODO: fill in]"
with open("outputs/my_project_update.md", "w") as f:
    f.write(update_content)
print("Project update saved to outputs/my_project_update.md")

# Summarize outputs
outputs = [
    "outputs/RESOLVER.md",
    "outputs/my_project_update.md",
]
print("\n=== NB08 Outputs ===")
for path in outputs:
    exists = os.path.exists(path)
    print(f"  {'[OK]' if exists else '[MISSING]'} {path}")

# Append to reflection log
def append_to_reflection(nb_id: str, nb_title: str, reflection: str, path: str = "outputs/reflection_log.json"):
    log = []
    if os.path.exists(path):
        with open(path) as f:
            log = json.load(f)
    log.append({
        "notebook": nb_id,
        "title": nb_title,
        "reflection": reflection,
        "timestamp": datetime.now().isoformat(),
    })
    with open(path, "w") as f:
        json.dump(log, f, indent=2)
    print(f"Reflection appended to {path}")

append_to_reflection(
    "08",
    "Project Integration",
    todo1_reflection if 'todo1_reflection' in dir() else "[TODO 1 not completed]",
)

tracker.report()

Project update saved to outputs/my_project_update.md

=== NB08 Outputs ===
  [OK] outputs/RESOLVER.md
  [OK] outputs/my_project_update.md
Reflection appended to outputs/reflection_log.json
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

